In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
DATA_DIR = "/kaggle/input/comment-category-prediction-challenge"
df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df=pd.read_csv(f"{DATA_DIR}/test.csv")

#Exploratory Data Analysis

print("Shape of dataset:", df.shape)

print("\nColumn names:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
df.head()

print("\nMissing values:")
print(df.isnull().sum())


sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

print("Number of duplicate rows:", df.duplicated().sum())

# Dropping duplicates if needed
df = df.drop_duplicates()

sns.countplot(x='label', data=df)
plt.title("Class Distribution")
plt.show()

print(df['label'].value_counts(normalize=True))

df_eda = df.copy()

df_eda['comment_length'] = df_eda['comment'].str.len()
df_eda['word_count'] = df_eda['comment'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,5))
sns.histplot(df_eda['comment_length'], bins=50, kde=True)
plt.title("Comment Length Distribution")
plt.xlabel("Comment Length")
plt.ylabel("Frequency")
plt.show()

df_eda['avg_word_length'] = df_eda['comment'].apply(lambda x: sum(len(w) for w in str(x).split()) / (len(str(x).split()) + 1))

df_eda[['word_count', 'avg_word_length']].describe()



label_counts=df["label"].value_counts().sort_index()
print(label_counts) #Label counts

label_counts.plot(kind="bar", title="Class Distribution")
plt.show()

df_eda.boxplot(column="comment_length", by="label", figsize=(8,4))
plt.title("Comment length by class")
plt.show()

As we can see from the bar graph, the dataset is highly imbalanced. Class 0 dominates, followed by class 2. 

From the boxplot, Classes 1 and 2 show the highest median comment length, followed by class 0 and class 3, which has the smallest median comment length. 

This means that comments of classes 1 and 2 tend to be longer on average. Class 3 comments are shorter.
Classes 0-2 show high variability in comment length while class 3 is more compact and consistent.

All classes have outliers, with class 0 having an extreme outlier.

There is significant overlap among classes 0,1,2. Class 3 shows lesser overlap.

In [ ]:
#Data Cleaning and Preprocessing


import re
import string
from sklearn.preprocessing import LabelEncoder

def engineer_features(df):

    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["unique_word_count"] = df["comment"].apply(lambda x: len(set(str(x).split())))
    df["avg_word_length"] = df["comment_length"] / (df["word_count"] + 1)

    df["num_uppercase"] = df["comment"].str.count(r"[A-Z]")
    df["uppercase_ratio"] = df["num_uppercase"] / (df["comment_length"] + 1)

    df["num_exclamation"] = df["comment"].str.count("!")
    df["num_question"] = df["comment"].str.count("\\?")
    df["num_punctuation"] = df["comment"].apply(lambda x: sum(1 for c in str(x) if c in string.punctuation))
    df["punct_ratio"] = df["num_punctuation"] / (df["comment_length"] + 1)

    df["digit_count"] = df["comment"].str.count(r"\d")
    df["digit_ratio"] = df["digit_count"] / (df["comment_length"] + 1)

    df["has_url"] = df["comment"].str.contains(r"http|www", regex=True).astype(int)
    df["has_email"] = df["comment"].str.contains(r"\S+@\S+", regex=True).astype(int)
    df["has_mention"] = df["comment"].str.contains(r"@\w+", regex=True).astype(int)
    df["has_hashtag"] = df["comment"].str.contains(r"#\w+", regex=True).astype(int)

    df["type_token_ratio"] = df["unique_word_count"] / (df["word_count"] + 1)
    df["caps_word_count"] = df["comment"].str.count(r"\b[A-Z]{2,}\b")
    df["elongated_words"] = df["comment"].str.count(r"(.)\1{2,}")

    return df


# Drop unused columns
df = df.drop(columns=["created_date", "post_id"])

X = df.drop(columns=["label"])
y = df["label"]

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train["comment"]=X_train["comment"].fillna("")
X_val["comment"]=X_val["comment"].fillna("")
test_df["comment"]=test_df["comment"].fillna("")


X_train=engineer_features(X_train)
X_val=engineer_features(X_val)
test_df=engineer_features(test_df)

corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=False, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

label_counts = y_train.value_counts(normalize=True)
print(label_counts)

ohe=OneHotEncoder(handle_unknown='ignore')

# Column groups
text_col = "comment"
categorical_cols = ["race", "religion", "gender", "disability"]
numeric_cols = [
    "emoticon_1", "emoticon_2", "emoticon_3",
    "upvote", "downvote", "if_1", "if_2"
]

for col in numeric_cols:
    X_train[col]=X_train[col].fillna(X_train[col].median())
    X_val[col]=X_val[col].fillna(X_val[col].median())
    test_df[col]=test_df[col].fillna(test_df[col].median())

for col in categorical_cols:
    X_train[col]=X_train[col].fillna("unknown")
    X_val[col]=X_val[col].fillna("unknown")
    test_df[col]=test_df[col].fillna("unknown")

for col in categorical_cols:
    
    combined = pd.concat([
        X_train[col],
        X_val[col],
        test_df[col]
    ], axis=0).astype(str)

    le = LabelEncoder()
    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str))
    X_val[col]   = le.transform(X_val[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))


As we can see from the correlation heatmap, there is very small, nearly negligible correlation between any 2 separate features in the original dataset (before feature engineering). The highest correlation between 2 separate features is shown by downvote and emoticon_3 group, followed by downvote and upvote group

In [ ]:
#Baseline Model
baseline_pipeline=Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        max_features=30000,
        stop_words="english"
    )),
    ("clf", LogisticRegression(
        max_iter=750,
        class_weight="balanced"
    ))
])

baseline_pipeline.fit(X_train["comment"], y_train)
baseline_preds=baseline_pipeline.predict(X_val["comment"])
print(classification_report(y_val, baseline_preds))

In [ ]:
#Logistic Regression Model for character based analysis using TF-IDF

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 4),
            min_df=10,
            max_features=30000,
            sublinear_tf=True
        ), text_col),
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ), categorical_cols)
    ],
    remainder="drop"
)

# Logistic Regression (multiclass)
clf = LogisticRegression(
    C=0.3,
    solver="saga",
    multi_class="multinomial",
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

# Full pipeline
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

# Train
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict_proba(X_val)
custom_preds=[]

for p in y_pred:
    if p[3]>0.30 and p[3]>p[2]*0.85:
        custom_preds.append(3)
    elif p[2]>0.30:
        custom_preds.append(2)
    elif p[1]>0.40:
        custom_preds.append(1)
    else:
        custom_preds.append(np.argmax(p))
custom_preds=np.array(custom_preds)

print(classification_report(y_val, custom_preds))


In [ ]:
#Logistic Regression Model for word based analysis using TF-IDF

# Preprocessing: WORD TF-IDF
preprocessor_word = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.8,
            max_features=30000,
            stop_words="english",
            sublinear_tf=True
        ), text_col),
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ), categorical_cols)
    ],
    remainder="drop"
)

# Logistic Regression
clf_word = LogisticRegression(
    C=0.3,
    solver="saga",
    multi_class="multinomial",
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

# Pipeline
model_word = Pipeline(steps=[
    ("preprocess", preprocessor_word),
    ("clf", clf_word)
])

# Train
model_word.fit(X_train, y_train)

# Evaluate (ARGMAX ONLY — no thresholds yet)
y_pred_word = model_word.predict(X_val)
print(classification_report(y_val, y_pred_word))


In [ ]:
#Logistic Regression Model for word based analysis using Count Vectorizer

from sklearn.feature_extraction.text import CountVectorizer

preprocessor_count = ColumnTransformer(
    transformers=[
        ("text", CountVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.9,
            max_features=50000
        ), text_col),
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
    ]
)

count_clf = LogisticRegression(
    C=0.5,
    solver="saga",
    max_iter=1250,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model_count = Pipeline([
    ("preprocess", preprocessor_count),
    ("clf", count_clf)
])

model_count.fit(X_train, y_train)
model_count_preds=model_count.predict(X_val)

print(classification_report(y_val, model_count_preds))

In [ ]:
#Logistic Regression Model for character based analysis using Count Vectorizer

preprocessor_char_count = ColumnTransformer(
    transformers=[
    ("text", CountVectorizer(
        analyzer="char",
        ngram_range=(3, 4),
        min_df=20,
        max_df=0.9,
        max_features=30000
    ), text_col),
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_cols)
    ]
    )

char_count_clf = LogisticRegression(
    C=0.5,
    solver="saga",
    max_iter=1250,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model_char_count=Pipeline([
    ("preprocess", preprocessor_char_count),
    ("clf", char_count_clf)
])

model_char_count.fit(X_train, y_train)
model_char_count_preds=model_char_count.predict(X_val)

print(classification_report(y_val,model_char_count_preds))

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline=Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3,4),
        max_features=30000
    )),
    ("clf", SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

sgd_pipeline.fit(X_train["comment"], y_train)
sgd_preds=sgd_pipeline.predict(X_val["comment"])

print(classification_report(y_val, sgd_preds))

In [ ]:
from sklearn.svm import LinearSVC

svm_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("svm", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

svm_clf.fit(X_train, y_train)

# SVM decision scores
svm_train_scores = svm_clf.decision_function(X_train)
svm_val_scores  = svm_clf.decision_function(X_val)

svm_val_preds= svm_clf.predict(X_val)

print(classification_report(y_val, svm_val_preds))

In [ ]:
import lightgbm as lgb

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [400, 600],
    "learning_rate": [0.03, 0.05],
    
}

p_char_train = model.predict_proba(X_train)
p_word_train = model_word.predict_proba(X_train)
p_count_train = model_count.predict_proba(X_train)
p_char_count_train = model_char_count.predict_proba(X_train)
p_baseline_train=baseline_pipeline.predict_proba(X_train["comment"])

X_lgb_train = np.hstack([
    p_char_train,
    p_word_train,
    p_count_train,
    p_char_count_train,
    p_baseline_train,
    svm_train_scores,
    X_train[numeric_cols].values
])

p_char_val = model.predict_proba(X_val)
p_word_val = model_word.predict_proba(X_val)
p_count_val = model_count.predict_proba(X_val)
p_char_count_val = model_char_count.predict_proba(X_val)
p_baseline_val=baseline_pipeline.predict_proba(X_val["comment"])

X_lgb_val = np.hstack([
    p_char_val,
    p_word_val,
    p_count_val,
    p_char_count_val,
    p_baseline_val,
    svm_val_scores,
    X_val[numeric_cols].values
])


lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=600,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=32,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=2.0,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    estimator=lgb_model,
    param_distributions=param_dist,
    n_iter=8,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_lgb_train, y_train)
best_lgb = search.best_estimator_

print("Best params:", search.best_params_)

best_lgb.fit(
    X_lgb_train, y_train,
    eval_set=[(X_lgb_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

lgb_val_proba = best_lgb.predict_proba(X_lgb_val)

# Blend LR ensemble + LightGBM
final_val_proba = 0.35 * (0.65 * p_count_val + 0.2 * p_char_count_val + 0.15 * p_baseline_val) + 0.65 * lgb_val_proba

y_val_pred = np.argmax(final_val_proba, axis=1)
print(classification_report(y_val, y_val_pred))


In [ ]:
# Test probabilities
sample = pd.read_csv(f"{DATA_DIR}/Sample.csv")


test_df["comment"] = test_df["comment"].fillna("")

p_char_test = model.predict_proba(test_df)
p_word_test = model_word.predict_proba(test_df)
p_count_test = model_count.predict_proba(test_df)
p_char_count_test = model_char_count.predict_proba(test_df)
p_baseline_test=baseline_pipeline.predict_proba(test_df["comment"])
svm_test_scores = svm_clf.decision_function(test_df)

X_lgb_test = np.hstack([
    p_char_test,
    p_word_test,
    p_count_test,
    p_char_count_test,
    p_baseline_test,
    svm_test_scores,
    test_df[numeric_cols].values
])

lgb_test_proba = best_lgb.predict_proba(X_lgb_test)

# Final blend
final_test_proba = 0.35 * (0.65 * p_count_test + 0.20 * p_char_count_test + 0.15 * p_baseline_test) + 0.65 * lgb_test_proba


final_test_proba[:, 2] *= 1.05
final_test_proba[:, 3] *= 1.35
final_test_proba = np.clip(final_test_proba, 1e-9, None)
final_test_proba /= final_test_proba.sum(axis=1, keepdims=True)

test_preds = np.argmax(final_test_proba, axis=1)


In [ ]:
print(test_df.shape)
print(np.bincount(test_preds))


In [ ]:
# Build submission
submission = sample.copy()
submission["label"] = test_preds.astype(int)

# Sanity checks
assert len(submission) == len(test_preds)
assert submission["label"].isna().sum() == 0
assert set(submission["label"]).issubset({0, 1, 2, 3})

# Distribution check (final)
print("Final prediction distribution:", np.bincount(test_preds))

# Save
submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully")

submission.head()
